# phys119-functions: verify an install

Run this **from the account being tested**, in a folder that does not
contain a copy of `phys119_functions.py`. Run every cell in order. The last
cell prints a single pass or fail verdict.

Two settings in the first cell:

- `EXPECTED` is the version to expect. Change it for a later release.
- `EXPECT_LOCATION` is where the package should be coming from:
  - `"shared"` after a system-wide install into the environment every
    account shares, which is what students get;
  - `"user"` after an install into one account only (under `~/.local`), which
    is how a single account is tested before going server-wide.

In [ ]:
EXPECTED = "1.3.0"
EXPECT_LOCATION = "shared"   # "shared" or "user"

import os
import site
import sys

import phys119_functions as pf

module_path = os.path.realpath(pf.__file__)
shared_prefix = os.path.realpath(sys.prefix)
user_site = os.path.realpath(site.getusersitepackages())

if module_path.startswith(user_site):
    location = "user"
elif module_path.startswith(shared_prefix):
    location = "shared"
elif os.path.dirname(module_path) == os.path.realpath(os.getcwd()):
    location = "working directory"
else:
    location = "somewhere else"

print("Account:           ", os.path.expanduser("~"))
print("Kernel executable: ", sys.executable)
print("Shared environment:", shared_prefix)
print("Module loaded from:", module_path)
print("That location is:  ", location, "(expected: " + EXPECT_LOCATION + ")")
print("Version:           ", getattr(pf, "__version__", "NONE"))
print()

problems = []

if getattr(pf, "__version__", None) != EXPECTED:
    problems.append(
        "Version is " + str(getattr(pf, "__version__", "missing"))
        + ", expected " + EXPECTED
    )

if location == EXPECT_LOCATION:
    print("OK: loaded from the", location, "location.")
elif location == "working directory":
    problems.append(
        "Loaded from the working directory. A stray phys119_functions.py "
        "sits next to this notebook."
    )
elif location == "user" and EXPECT_LOCATION == "shared":
    problems.append(
        "Loaded from this account's personal packages, which is hiding the "
        "shared install. Run: pip uninstall --user phys119-functions"
    )
elif location == "shared" and EXPECT_LOCATION == "user":
    problems.append(
        "Loaded from the shared environment, so the per-account install "
        "under test is not the copy being used."
    )
else:
    problems.append("Loaded from an unexpected place: " + module_path)

## Values

Checked against numpy, so this catches a broken install rather than a
mistyped constant.

In [ ]:
import numpy as np

from phys119_functions import *

sample = [10.1, 10.3, 9.8]

checks = [
    ("t_score(10, 2, 12, 3)", t_score(10, 2, 12, 3), 0.5547001962252291),
    ("mean(sample)", mean(sample), float(np.mean(sample))),
    ("standard_deviation(sample)", standard_deviation(sample), float(np.std(sample, ddof=1))),
    ("standard_unc_of_mean(sample)", standard_unc_of_mean(sample),
     float(np.std(sample, ddof=1) / np.sqrt(len(sample)))),
]

for label, got, want in checks:
    ok = got is not None and abs(got - want) < 1e-12 and type(got) is float
    print(("OK  " if ok else "FAIL"), label, "->", got)
    if not ok:
        problems.append(label + " gave " + repr(got) + ", expected " + repr(want))

## What `import *` brings in

Expect the eight function names, `Phys119Error` and `np`, and nothing else.

In [ ]:
namespace = {}
exec("from phys119_functions import *", namespace)
exported = sorted(name for name in namespace if not name.startswith("__"))

print(exported)
if exported != ["Phys119Error", "autofit", "manual_fit", "mean", "np",
                "plot_data", "plot_residuals", "standard_deviation",
                "standard_unc_of_mean", "t_score"]:
    problems.append("import * exported " + repr(exported))

## Error behaviour

The next cell should show a red box about needing at least 2 measurements,
with **no traceback**. The `try` is here only so this notebook reaches its
verdict: an error stops the cell, so without it nothing below would run.

In [ ]:
try:
    standard_deviation([4.2])
    problems.append("an impossible call did not stop the cell")
except Phys119Error:
    pass

## Plotting

The next cell should draw one figure: five blue points with error bars, a
red line, and a residuals panel underneath whose axis reads "residual =
data - model". The title shows $y = 2x$ as maths, and the legend lists
Data, Model (y=mx+b), m = 2.00, b = 0.00 and $\chi^2$.

In [ ]:
import matplotlib

xs = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
ys = np.array([2.1, 3.9, 6.2, 7.8, 10.1])
dys = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

print("matplotlib", matplotlib.__version__)
plot_data(xs, ys, dys, m=2, b=0, residuals=True, chi2=True,
          title=r"Check: $y = 2x$")

`autofit` should draw a second figure and find the same line as numpy's
weighted fit. Then a LaTeX mistake in a label should give a red box, with
no traceback and no figure.

In [ ]:
fit_m, fit_b = autofit(xs, ys, dys)
want_m, want_b = np.polyfit(xs, ys, 1, w=1 / dys)
ok = abs(fit_m - want_m) < 1e-9 and abs(fit_b - want_b) < 1e-9
print(("OK  " if ok else "FAIL"), "autofit ->", fit_m, fit_b)
if not ok:
    problems.append(f"autofit gave m={fit_m}, b={fit_b}; numpy gives {want_m}, {want_b}")

try:
    plot_data(xs, ys, dys, title=r"$\Deltax$")
    problems.append("a LaTeX mistake in a title did not stop the cell")
except Phys119Error:
    pass

## manual_fit

This needs the anywidget package in the same environment. The next cell
should show the interactive fit: a plot with a residuals panel, and below
it a slider and a number box for each of m and b. **Drag a slider**: the
line and the residuals should move with it. Then type a value in a box and
press Enter.

The fit is remembered in a small file beside this notebook,
`.verify_install.ipynb.manual_fits.json`, which can be deleted afterwards.

In [ ]:
try:
    import anywidget
    widget_path = os.path.realpath(anywidget.__file__)
    if widget_path.startswith(user_site):
        widget_location = "user"
    elif widget_path.startswith(shared_prefix):
        widget_location = "shared"
    else:
        widget_location = "somewhere else"
    print("anywidget", anywidget.__version__, "from the", widget_location, "location:")
    print("  ", widget_path)
    if EXPECT_LOCATION == "shared" and widget_location != "shared":
        problems.append(
            "anywidget is loaded from the " + widget_location + " location, so "
            "students, who lack that copy, would not have it. Install it in the "
            "shared environment and remove this account's copy."
        )
except ImportError as error:
    problems.append("anywidget is not installed, so manual_fit cannot work: " + str(error))

manual_fit(xs, ys, dys)

In [ ]:
import re

fits = sorted(name for name in globals() if re.fullmatch(r"manual_fit\d+", name))
if fits:
    fit = globals()[fits[-1]]
    print("OK  ", fits[-1], "->", fit)
else:
    problems.append("manual_fit did not create a manual_fit1 (or later) name")

## Verdict

In [ ]:
print()
if problems:
    print("FAILED")
    for problem in problems:
        print(" -", problem)
else:
    scope = "for all users" if EXPECT_LOCATION == "shared" else "for this account"
    print("ALL CHECKS PASSED")
    print("phys119-functions", EXPECTED, "is installed", scope, "and behaves correctly.")